In [ ]:
import pickle
from transformers import pipeline
from pathlib import Path
from collections import defaultdict
from PIL import Image
import numpy as np
from tqdm import tqdm
from videoutils import make_animation
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd 


In [ ]:
train_videos = pickle.load(open("train_structure.pkl", "rb"))
test_videos = pickle.load(open("test_structure.pkl", "rb"))

In [ ]:
pipe1 = pipeline('image-classification', model="dima806/crime_cctv_image_detection", device=0)
pipe2 = pipeline('image-classification', model="dima806/crime_type_cctv_image_detection", device=0)

In [ ]:
def pipe(image, pipe1, pipe2):
    res1 = pipe1(image)

    crime_score = 0
    for r in res1:
        if r["label"] == "Crime":
            crime_score = r["score"]
            break
    
    if crime_score > 0.5:
        res2 = pipe2(image)
        predicted_label = res2[0]["label"] if len(res2) > 0 else "Unknown"
        return crime_score, predicted_label
    
    return crime_score, "Normal"

In [ ]:
def analyze_video(video_frames, pipe1, pipe2):    
    crime_count = 0
    crimes = {}

    for frame_path in video_frames:
        image = Image.open(frame_path)

        crime_score, label = pipe(image, pipe1, pipe2)

        if crime_score > 0.5:
            crime_count += 1
            crimes[label] = crimes.get(label, 0) + 1

    return crime_count, crimes

In [ ]:
for i in train_videos:
    if i != "NormalVideos":
        print(i)

In [ ]:
crime_results = []

for class_name, videos in train_videos.items():
    # only crime
    if class_name == "NormalVideos":
        continue

    for video_name, info in videos.items():
        frames = info["frames"]

        crime_count, crimes = analyze_video(frames, pipe1, pipe2)

        row = {"vid_name": video_name}

        for label, count in crimes.items():
            row[label] = count

        if len(crimes) > 0:
            max_crime = max(crimes, key=crimes.get)
            max_value = crimes[max_crime]
        else:
            max_crime = "None"
            max_value = 0

        row["max_crime"] = max_crime
        row["max_value"] = max_value
        row["TrueLabel"] = class_name

        crime_results.append(row)
df = pd.DataFrame(crime_results)


In [ ]:
df = pd.DataFrame(crime_results)
df = df.fillna(0)

print(df.head())

In [ ]:
df.to_csv("crime_results.csv", index=False)

In [ ]:
for class_name, videos in train_videos.items():
    if class_name == "NormalVideos":
        continue
        
    for video_name, info in videos.items():
        frames = info["frames"]
        
        crime_count, crimes = analyze_video(frames, pipe1, pipe2)

        print(crime_count)
        print(crimes)

        

In [ ]:
crime_results